# RWJ GME PubMed — v18

Searches PubMed by trainee author, matches author-specific affiliations against the five-column roster, supports Excel review, and generates a Word newsletter plus a one-sheet filterable Excel publication index.

**Dates:** Defaults to the previous calendar month. Edit `start_date` and `end_date` in the settings cell for any custom range. Both start and end dates are included.

**Workflow:** Run All → save any review decisions in the internal Excel workbook → rerun only the final export cell to regenerate the final Word report **and** the distributable Excel index.

V15: centered, larger department headings; Page X of Y footer; program-coverage diagnostics. Automatic PDF conversion removed to avoid delays; export a PDF from Microsoft Word when needed.


# USER SETTINGS — EDIT HERE BEFORE RUNNING

**These are the only three settings you normally need to change.**

- **`INPUT_FILE`** — name or full path of your trainee Excel roster.
- **`START_DATE`** — first day to include (based on the date the article was added to PubMed).
- **`END_DATE`** — last day to include (inclusive; based on the date the article was added to PubMed).

**Default:** The two date values automatically cover the entire **previous calendar month**, whenever you run the notebook. To search a different period, replace **both** date values with dates in quotes, e.g. `"2026-01-01"` and `"2026-06-30"`. No switch is needed.

The rest of the notebook uses these settings automatically. Run **All** for the PubMed search and draft; after reviewing and saving the `_FOR_REVIEW.xlsx` workbook, run **only the final export cell** again.

In [13]:
# ==========================================================
# USER SETTINGS — THESE ARE THE THREE VALUES YOU MAY CHANGE
# ==========================================================

INPUT_FILE = "rutgers_gme_trainee_list_v05.xlsx"

# Defaults to the first and last day of the PREVIOUS month.
# For a custom range, replace EACH expression below with a
# date in quotes, e.g. "2026-01-01" and "2026-06-30".
from datetime import date, timedelta
_previous_month_last_day = date.today().replace(day=1) - timedelta(days=1)
#START_DATE = _previous_month_last_day.replace(day=1)
#END_DATE = _previous_month_last_day

# Custom dates for three-month analysis
START_DATE = "2026-08-01"
END_DATE = "2026-09-30"

print(f"Roster file: {INPUT_FILE}")
print(f"PubMed entry dates: {START_DATE} through {END_DATE} (inclusive)")

Roster file: rutgers_gme_trainee_list_v05.xlsx
PubMed entry dates: 2026-08-01 through 2026-09-30 (inclusive)


In [14]:
import time
run_start = time.perf_counter()
print("GME PubMed run started")

GME PubMed run started


In [15]:
import re
import time
import calendar
import unicodedata
from collections import defaultdict
from datetime import date
from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from docx import Document
from docx.shared import Inches, Pt

## 1 — Load roster and validate reporting dates

Reads the **USER SETTINGS** at the top of this notebook. The remaining technical defaults ordinarily need no editing.

In [16]:
# Read the three user-configurable values defined above.
start_date = START_DATE
end_date = END_DATE

def parse_date(value):
    if isinstance(value, date):
        return value
    return date.fromisoformat(str(value).strip())

start_date = parse_date(start_date)
end_date = parse_date(end_date)
if start_date > end_date:
    raise ValueError("start_date must be on or before end_date")

ROSTER_FILE = Path(INPUT_FILE).expanduser()
OUTPUT_FOLDER = Path.cwd()
NEWSLETTER_TITLE = "Rutgers Health GME Academic Newsletter"
FONT_NAME = "Calibri"
FONT_SIZE = 10
PAGE_MARGIN_INCHES = 0.8
DATE_TYPE = "edat"      # As in v18.02: newly entered into PubMed during the month
CONTACT_EMAIL = "rhsavel@gmail.com"
NAMES_PER_QUERY = 35    # POST keeps the long OR expression out of the URL
MAX_HITS_PER_QUERY = 8000  # Stop rather than silently truncate a broad search
API_DELAY_SECONDS = 0.38
MAX_MANUAL_REVIEW = 12

ROSTER_COLUMNS = ["name", "trainee type (resident/fellow)", "name of program", "Department", "Sponsoring entity/hospital"]
roster = pd.read_excel(ROSTER_FILE, sheet_name="main list", dtype=str).fillna("")
if not set(ROSTER_COLUMNS).issubset(roster.columns):
    raise ValueError(f"Missing roster columns: {set(ROSTER_COLUMNS) - set(roster.columns)}")
roster = roster[ROSTER_COLUMNS].copy()
for col in ROSTER_COLUMNS:
    roster[col] = roster[col].astype(str).str.strip()
roster = roster[roster["name"].ne("")].drop_duplicates(subset=ROSTER_COLUMNS).reset_index(drop=True)
roster = roster.rename(columns={"name":"Trainee Name", "trainee type (resident/fellow)":"Trainee Type", "name of program":"Program", "Department":"Department", "Sponsoring entity/hospital":"Home Institution"})
roster["Program Key"] = roster["Home Institution"] + " | " + roster["Department"] + " | " + roster["Program"]
PROGRAM_CHOICES = sorted(roster["Program Key"].unique(), key=str.casefold)
PROGRAM_LOOKUP = roster.drop_duplicates("Program Key").set_index("Program Key")[["Home Institution", "Department", "Program"]].to_dict("index")
month_start = start_date
month_end = end_date
period_tag = f"{start_date:%Y-%m-%d}_to_{end_date:%Y-%m-%d}"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Loaded {len(roster):,} trainees; {len(PROGRAM_CHOICES)} institution/department/program combinations")
print(f"Articles added to PubMed: {start_date} to {end_date} (inclusive)")

Loaded 1,424 trainees; 55 institution/department/program combinations
Articles added to PubMed: 2026-08-01 to 2026-09-30 (inclusive)


## 2 — Build PubMed author queries
Each trainee generates an initial-and-surname `[au]` query. Multiple names are combined with `OR` in POST requests, with the date range applied by PubMed. The resulting article IDs are deduplicated **before** EFetch. Query batching is separate from EFetch batching.

In [17]:
def plain(value):
    value = unicodedata.normalize("NFKD", str(value))
    return "".join(c for c in value if not unicodedata.combining(c)).casefold()

def words(value):
    return re.findall(r"[a-z0-9]+", plain(value))

def author_parts(full_name):
    # Default: final token surname. Preserve common prefixes and hyphenated surnames.
    # Ambiguous multi-word surnames are flagged by the review logic.
    chunks = str(full_name).split()
    if len(chunks) < 2:
        return None
    surname = chunks[-1]
    first = chunks[0]
    if len(chunks) >= 3 and chunks[-2].casefold() in {"de", "del", "van", "von", "da", "dos", "bin", "al"}:
        surname = " ".join(chunks[-2:])
    return surname, first

def author_query(name):
    parts = author_parts(name)
    if not parts:
        return None
    surname, first = parts
    initial = words(first)[0][:1] if words(first) else ""
    surname = " ".join(words(surname))
    return f'"{surname} {initial}"[au]' if surname and initial else None

terms = sorted({q for name in roster["Trainee Name"] if (q := author_query(name))})
query_batches = [terms[i:i + NAMES_PER_QUERY] for i in range(0, len(terms), NAMES_PER_QUERY)]
print(f"{len(terms)} unique author terms in {len(query_batches)} PubMed searches")

1361 unique author terms in 39 PubMed searches


## 3 — ESearch and EFetch (adapted from v18.02)
Uses the same NCBI E-utilities endpoints and the same safety principle: stop on unexpectedly large queries, never silently truncate. The ESearch query uses POST to accommodate longer author-name lists.

In [18]:
BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
session = requests.Session()
session.headers.update({"User-Agent": "RWJGMEPubMed-v06/1.0 (academic newsletter)"})
common_params = {"tool": "RWJGMEPubMed"}
if CONTACT_EMAIL.strip():
    common_params["email"] = CONTACT_EMAIL.strip()

def get_api(endpoint, params, post=False):
    method = session.post if post else session.get
    # Basic retry for transient NCBI errors and rate limits.
    for attempt in range(4):
        try:
            response = method(BASE + endpoint, data={**common_params, **params} if post else None,
                              params=None if post else {**common_params, **params}, timeout=75)
            if response.status_code in (429, 500, 502, 503, 504):
                if attempt < 3:
                    time.sleep(2 ** (attempt + 1))
                    continue
            response.raise_for_status()
            return response
        except requests.RequestException:
            if attempt == 3:
                raise
            time.sleep(2 ** (attempt + 1))

# Large searches are automatically divided into smaller groups.
# If even one author produces too many results, divide that author's date range.
def search_author_group(batch, date_begin, date_finish):
    search_params = {
        "db": "pubmed", "term": " OR ".join(batch), "datetype": DATE_TYPE,
        "mindate": date_begin.strftime("%Y/%m/%d"),
        "maxdate": date_finish.strftime("%Y/%m/%d"),
        "retmode": "json", "retmax": 0,
    }
    time.sleep(API_DELAY_SECONDS)
    result = get_api("esearch.fcgi", search_params, post=True).json()["esearchresult"]
    count = int(result["count"])

    if count > MAX_HITS_PER_QUERY:
        if len(batch) > 1:
            midpoint = len(batch) // 2
            return (search_author_group(batch[:midpoint], date_begin, date_finish)
                    | search_author_group(batch[midpoint:], date_begin, date_finish))
        if date_begin < date_finish:
            half_days = (date_finish - date_begin).days // 2
            middle = date_begin + timedelta(days=half_days)
            return (search_author_group(batch, date_begin, middle)
                    | search_author_group(batch, middle + timedelta(days=1), date_finish))
        raise RuntimeError(
            f"One author search returned {count:,} records on {date_begin}. "
            "Cannot retrieve all results safely; no records were silently discarded."
        )

    found_ids = []
    for offset in range(0, count, 500):
        time.sleep(API_DELAY_SECONDS)
        page = get_api("esearch.fcgi", {
            **search_params, "retstart": offset, "retmax": 500
        }, post=True).json()["esearchresult"]
        found_ids.extend(page["idlist"])
    if len(found_ids) != count:
        raise RuntimeError(
            f"Expected {count} PubMed IDs, received {len(found_ids)}. "
            "The search stopped instead of omitting results."
        )
    return set(found_ids)

pmid_set = set()
for number, batch in enumerate(query_batches, 1):
    pmid_set.update(search_author_group(batch, month_start, month_end))
    if number % 5 == 0 or number == len(query_batches):
        print(f"Author search batch {number}/{len(query_batches)}; {len(pmid_set):,} unique candidate PMIDs")
pmids = sorted(pmid_set)
print(f"PubMed found {len(pmids):,} unique candidate articles for the selected dates")


Author search batch 5/39; 838 unique candidate PMIDs
Author search batch 10/39; 7,152 unique candidate PMIDs
Author search batch 15/39; 9,757 unique candidate PMIDs
Author search batch 20/39; 22,223 unique candidate PMIDs
Author search batch 25/39; 22,785 unique candidate PMIDs
Author search batch 30/39; 24,271 unique candidate PMIDs
Author search batch 35/39; 26,524 unique candidate PMIDs
Author search batch 39/39; 34,193 unique candidate PMIDs
PubMed found 34,193 unique candidate articles for the selected dates


## 4 — Parse articles and match the author to the actual roster
Affiliations are collected **per author**, not only at the article level. Matching a name means it is a *candidate*, not proof the person is your trainee. Only a full-name match with supporting institutional evidence is automatically labeled **Matched**; all other candidates require review.

In [19]:
def element_text(element):
    if element is None:
        return ""
    return "".join(element.itertext()).strip()

def publication_date(article):
    for path in ["MedlineCitation/Article/ArticleDate", "MedlineCitation/Article/Journal/JournalIssue/PubDate"]:
        item = article.find(path)
        if item is not None:
            bits = [element_text(item.find(x)) for x in ["Year", "Month", "Day"]]
            if any(bits):
                return " ".join(b for b in bits if b)
            medline = element_text(item.find("MedlineDate"))
            if medline:
                return medline
    return ""

In [20]:
# Site terms are contextual signals, not exclusion rules. A published paper may
# have an outside affiliation, especially for residents working with other groups.
INSTITUTION_HINTS = {
    "Rutgers New Jersey Medical School": ["new jersey medical school", "njms", "rutgers", "university hospital newark"],
    "Rutgers Robert Wood Johnson Medical School": ["robert wood johnson medical school", "rwjms", "rutgers", "robert wood johnson university hospital"],
    "Jersey City Medical Center": ["jersey city medical center", "jersey city", "rwj barnabas"],
    "Newark Beth Israel Medical Center": ["newark beth israel", "beth israel medical center", "rwj barnabas"],
    "Community Medical Center": ["community medical center", "toms river", "rwj barnabas"],
    "Monmouth Medical Center": ["monmouth medical center", "long branch", "rwj barnabas"],
    "Cooperman Barnabas Medical Center": ["cooperman barnabas", "saint barnabas medical center", "st barnabas medical center"],
    "Trinitas Regional Medical Center": ["trinitas", "elizabeth new jersey"],
    "Robert Wood Johnson University Hospital Somerset": ["robert wood johnson university hospital somerset", "somerville"],
    "RWJBarnabas Health — Integrated Pathology Residency": ["rwj barnabas", "cooperman", "newark beth israel", "monmouth"]
}

def article_authors(article):
    detail = article.find("MedlineCitation/Article")
    if detail is None:
        return []
    results = []
    for node in detail.findall("AuthorList/Author"):
        given = " ".join((node.findtext("ForeName") or "").split())
        last = " ".join((node.findtext("LastName") or "").split())
        if not last:
            continue
        affs = [" ".join((a.text or "").split()) for a in node.findall("AffiliationInfo/Affiliation")]
        results.append({"name": f"{given} {last}".strip(), "given":given, "last":last,
                        "initials":node.findtext("Initials") or "", "affiliations":affs})
    return results

def candidate_match(trainee, author):
    """Return (possible, first-name-confirmed); reject conflicting expanded given names.

    PubMed may publish abbreviated middle initials. A trainee's expanded middle
    name should not be allowed to contradict an author's expanded middle name.
    """
    parts = author_parts(trainee)
    if not parts:
        return False, False
    surname, first = parts
    if words(surname) != words(author["last"]):
        return False, False
    target = words(trainee)
    given = words(author["given"])
    ini = words(author["initials"])
    first_token = words(first)[0] if words(first) else ""
    author_first = given[0] if given else (ini[0] if ini else "")
    if not first_token or not author_first or first_token[0] != author_first[0]:
        return False, False
    first_exact = bool(given and first_token == given[0])
    if not first_exact:
        return True, False
    # Compare extra spelled-out given names when they are present on both sides.
    # Initials alone never justify a contradiction.
    trainee_parts = str(trainee).split()
    trainee_given = words(" ".join(trainee_parts[:-1]))
    if len(given) > 1 and len(trainee_given) > 1:
        for a, b in zip(trainee_given[1:], given[1:]):
            if a[0] != b[0] or (len(a)>1 and len(b)>1 and a != b):
                return False, False
    return True, True

# A home-site-specific string is strong evidence; system-wide branding alone
# is helpful context, but not proof of employment in one training program.
SITE_SPECIFIC_HINTS = {
    "Rutgers New Jersey Medical School": ["new jersey medical school", "njms", "university hospital newark"],
    "Rutgers Robert Wood Johnson Medical School": ["robert wood johnson medical school", "rwjms", "robert wood johnson university hospital"],
    "Jersey City Medical Center": ["jersey city medical center"],
    "Newark Beth Israel Medical Center": ["newark beth israel"],
    "Community Medical Center": ["community medical center", "toms river"],
    "Monmouth Medical Center": ["monmouth medical center", "long branch"],
    "Cooperman Barnabas Medical Center": ["cooperman barnabas", "saint barnabas medical center", "st barnabas medical center"],
    "Trinitas Regional Medical Center": ["trinitas regional medical center", "trinitas regional"],
    "Robert Wood Johnson University Hospital Somerset": ["robert wood johnson university hospital somerset"],
    "RWJBarnabas Health — Integrated Pathology Residency": ["rwj barnabas health", "rwj barnabas"]
}
# Require distinctive, recognizable local institutional signals. Generic words
# such as "Somerset", "University Hospital", "Monmouth" and "Newark" by themselves
# can occur at unrelated institutions worldwide and must not trigger review.
REGIONAL_HINTS = ["rutgers", "rwj", "rwj barnabas", "new jersey", "jersey city",
                  "toms river", "cooperman", "trinitas", "robert wood johnson",
                  "hackensack meridian", "new brunswick nj", "newark nj", "newark new jersey"]

def phrase_found(text, phrase):
    needle = " ".join(words(phrase))
    return bool(needle and f" {needle} " in f" {text} ")

def institution_strength(affiliations, home):
    text = " ".join(words(" ".join(affiliations)))
    site_clues = SITE_SPECIFIC_HINTS.get(home, [home])
    if any(phrase_found(text, clue) for clue in site_clues):
        # Beware the generic name 'Community Medical Center' on an unrelated
        # institution; require an NJ/RWJ or Toms River clue for auto-credit.
        if home == "Community Medical Center" and not any(phrase_found(text, z) for z in
                ["toms river", "new jersey", "nj", "rwj", "barnabas"]):
            return "none"
        if home == "Monmouth Medical Center" and not any(phrase_found(text, z) for z in
                ["monmouth medical center", "long branch new jersey", "long branch nj", "rwj"]):
            return "none"
        return "site"
    if any(phrase_found(text, clue) for clue in REGIONAL_HINTS):
        return "regional"
    return "none"

# Index trainees on surname + first initial to avoid checking all 1,339 for each paper.
roster_index = defaultdict(list)
for trainee in roster.to_dict("records"):
    parts = author_parts(trainee["Trainee Name"])
    if parts:
        surname, given = parts
        roster_index[(tuple(words(surname)), words(given)[0][:1])].append(trainee)

COLUMNS = ["Status","Reason","PMID","Trainee Name","Trainee Type","Home Institution",
           "Department","Program","Program Key","Title","Authors","Journal","Publication Date",
           "Matched Author","Author Affiliations","DOI","PubMed Link","DOI Link"]
rows = []
for start in range(0, len(pmids), 150):
    time.sleep(API_DELAY_SECONDS)
    batch = pmids[start:start+150]
    xml_response = get_api("efetch.fcgi", {"db":"pubmed", "id":",".join(batch), "retmode":"xml"})
    root = ET.fromstring(xml_response.content)
    for article in root.findall("PubmedArticle"):
        med = article.find("MedlineCitation")
        if med is None:
            continue
        pmid = med.findtext("PMID") or ""
        detail = med.find("Article")
        if detail is None:
            continue
        authors = article_authors(article)
        doi = next((x.text for x in article.findall(".//ArticleId") if x.get("IdType") == "doi" and x.text), "")
        for author in authors:
            key = (tuple(words(author["last"])), words(author["given"])[0][:1] if words(author["given"]) else (words(author["initials"])[0][:1] if words(author["initials"]) else ""))
            for trainee in roster_index.get(key, []):
                possible, full = candidate_match(trainee["Trainee Name"], author)
                if not possible:
                    continue
                # Only a full first-name match is actionable. Reject initials-only
                # candidates from the workbook; PubMed remains broadly searched.
                if not full:
                    continue
                strength = institution_strength(author["affiliations"], trainee["Home Institution"])
                matched = strength == "site"
                reason = ("Full first name and author-specific home-site evidence" if matched
                          else "Full first name; regional/system affiliation requires verification" if strength == "regional"
                          else "Full first name; outside or absent affiliation (audit only)")
                status = "Matched" if matched else ("Needs Review" if strength == "regional" else "Archived")
                rows.append({"Status":status, "Reason":reason,
                             "PMID":pmid, "Trainee Name":trainee["Trainee Name"], "Trainee Type":trainee["Trainee Type"],
                             "Home Institution":trainee["Home Institution"], "Department":trainee["Department"],
                             "Program":trainee["Program"], "Program Key":trainee["Program Key"],
                             "Title":" ".join("".join(detail.find("ArticleTitle").itertext()).split()) if detail.find("ArticleTitle") is not None else "",
                             "Authors":"; ".join(a["name"] for a in authors),
                             "Journal":detail.findtext("Journal/Title") or "", "Publication Date":publication_date(article),
                             "Matched Author":author["name"], "Author Affiliations":" | ".join(author["affiliations"]),
                             "DOI":doi, "PubMed Link":f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                             "DOI Link":f"https://doi.org/{doi}" if doi else ""})
    if start % 750 == 0:
        print(f"EFetch: {min(start+150,len(pmids)):,}/{len(pmids):,} records requested")

df = pd.DataFrame(rows, columns=COLUMNS).drop_duplicates(subset=["PMID","Trainee Name","Program Key"])
matched_df = df[df["Status"].eq("Matched")].copy()
review_candidates = df[df["Status"].eq("Needs Review")].copy()
# Review only full first-name matches with identifiable New Jersey/regional evidence.
# Outside or unsupported full-name matches stay in Other Full Name Hits for audit.
# A fixed queue cap does not imply every archived candidate is false.
review_candidates["Evidence Rank"] = review_candidates["Author Affiliations"].fillna("").map(
    lambda x: sum(phrase in " ".join(words(x)) for phrase in
                  ["rutgers", "new jersey medical school", "robert wood johnson",
                   "rwj", "barnabas", "jersey city medical center", "newark beth israel",
                   "monmouth medical center", "university hospital"]))
review_candidates = review_candidates.sort_values(["Evidence Rank", "Trainee Name", "PMID"],
                                                  ascending=[False, True, True], kind="stable")
needs_review = review_candidates.head(MAX_MANUAL_REVIEW).drop(columns=["Evidence Rank"])
review_overflow = review_candidates.iloc[MAX_MANUAL_REVIEW:].drop(columns=["Evidence Rank"])
archive = df[df["Status"].eq("Archived")].copy()
print(f"Full-name candidate pairs: {len(df):,}; automatic matches: {len(matched_df):,}; "
      f"manual review: {len(needs_review):,}; additional plausible candidates: {len(review_overflow):,}; "
      f"other full-name candidates archived: {len(archive):,}")
if len(review_overflow):
    print("IMPORTANT: Additional plausible candidates are in Review Overflow. "
          "They will NOT enter the report without an explicit review decision.")

EFetch: 150/34,193 records requested
EFetch: 900/34,193 records requested
EFetch: 1,650/34,193 records requested
EFetch: 2,400/34,193 records requested
EFetch: 3,150/34,193 records requested
EFetch: 3,900/34,193 records requested
EFetch: 4,650/34,193 records requested
EFetch: 5,400/34,193 records requested
EFetch: 6,150/34,193 records requested
EFetch: 6,900/34,193 records requested
EFetch: 7,650/34,193 records requested
EFetch: 8,400/34,193 records requested
EFetch: 9,150/34,193 records requested
EFetch: 9,900/34,193 records requested
EFetch: 10,650/34,193 records requested
EFetch: 11,400/34,193 records requested
EFetch: 12,150/34,193 records requested
EFetch: 12,900/34,193 records requested
EFetch: 13,650/34,193 records requested
EFetch: 14,400/34,193 records requested
EFetch: 15,150/34,193 records requested
EFetch: 15,900/34,193 records requested
EFetch: 16,650/34,193 records requested
EFetch: 17,400/34,193 records requested
EFetch: 18,150/34,193 records requested
EFetch: 18,900/34,

In [21]:

# PROGRAM-COVERAGE AUDIT — a zero-publication program is worth inspecting, not
# automatically evidence of missed papers. This never changes accepted matches.
coverage = roster.groupby(["Home Institution", "Department", "Program"], dropna=False).size().rename("Trainees").reset_index()
if len(df):
    all_candidates = df.groupby(["Home Institution", "Department", "Program"], dropna=False).agg(
        Full_Name_Candidates=("PMID", "nunique"),
        Trainee_Candidates=("Trainee Name", "nunique")
    ).reset_index()
    coverage = coverage.merge(all_candidates, how="left", on=["Home Institution", "Department", "Program"])
else:
    coverage["Full_Name_Candidates"] = 0
    coverage["Trainee_Candidates"] = 0
confirmed = matched_df.groupby(["Home Institution", "Department", "Program"], dropna=False).agg(
    Auto_Accepted_Papers=("PMID", "nunique")
).reset_index() if len(matched_df) else pd.DataFrame(columns=["Home Institution", "Department", "Program", "Auto_Accepted_Papers"])
coverage = coverage.merge(confirmed, how="left", on=["Home Institution", "Department", "Program"])
for col in ["Full_Name_Candidates", "Trainee_Candidates", "Auto_Accepted_Papers"]:
    coverage[col] = coverage[col].fillna(0).astype(int)
coverage["Notice"] = coverage.apply(lambda r: "No automatically verified papers — investigate if unexpected"
    if r["Auto_Accepted_Papers"] == 0 else "", axis=1)
nb_im = coverage.loc[(coverage["Home Institution"] == "Rutgers Robert Wood Johnson Medical School") &
                     (coverage["Program"] == "Internal Medicine")]
if not nb_im.empty:
    entry = nb_im.iloc[0]
    print("RWJ New Brunswick Internal Medicine: "
          f"{entry['Trainees']} roster trainees; "
          f"{entry['Full_Name_Candidates']} unique full-name candidate papers; "
          f"{entry['Auto_Accepted_Papers']} auto-accepted papers.")
    if entry["Auto_Accepted_Papers"] == 0:
        print("CHECK: No confirmed RWJ New Brunswick IM articles added to PubMed during this period. "
              "Inspect 'Program Coverage' and 'All Full Name Hits'; do not assume zero publications means none exist.")


RWJ New Brunswick Internal Medicine: 74 roster trainees; 32 unique full-name candidate papers; 2 auto-accepted papers.


## 5 — Excel review file

**Needs Review** holds up to 15 prioritized regionally plausible cases. For a confirmed author, choose Include? = Yes and assign their program, or choose **Other** or **Multiple Hospitals** when appropriate. **Review Overflow** holds any further plausible cases, clearly separated rather than lost. **Other Full Name Hits** contains full-name matches with no regional affiliation evidence for archival checking. Initial-and-surname-only matches are not reviewable evidence and are omitted. Choose **Yes** and a program only for papers you can verify.


In [22]:
# Special categories used when a validated trainee author cannot be assigned
# to exactly one existing home-program choice. The usual choices remain intact.
REVIEW_ASSIGNMENT_CHOICES = PROGRAM_CHOICES + ["Other", "Multiple Hospitals"]
run_day = date.today().isoformat()
for run_number in range(1, 1000):
    run_stem = f"rwj_gme_pubmed_v18_{period_tag}_{run_day}_v{run_number:02d}"
    if not any(OUTPUT_FOLDER.glob(f"{run_stem}*")):
        break
else:
    raise RuntimeError("Too many reports for this day")
excel_path = OUTPUT_FOLDER / f"{run_stem}_FOR_REVIEW.xlsx"
with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    wb = writer.book
    option_ws = wb.add_worksheet("Dropdown Choices")
    option_ws.write(0,0,"Home institution | Department | Program")
    for index, item in enumerate(REVIEW_ASSIGNMENT_CHOICES, start=1):
        option_ws.write(index, 0, item)
    option_ws.hide()
    wb.define_name("ProgramOptions", f"='Dropdown Choices'!$A$2:$A${len(REVIEW_ASSIGNMENT_CHOICES)+1}")
    for sheet_name, frame in [
        ("By Program", matched_df), ("Matched Articles", matched_df),
        ("Needs Review", needs_review.assign(**{"Include?":"", "Program to Assign":""})),
        ("Review Overflow", review_overflow.assign(**{"Include?":"", "Program to Assign":""})),
        ("Other Full Name Hits", archive),
        ("All Full Name Hits", df), ("Program Coverage", coverage)]:
        frame.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]
        hdr = wb.add_format({"bold":True,"font_color":"white","bg_color":"#17365D","text_wrap":True})
        wrap = wb.add_format({"text_wrap":True,"valign":"top"})
        for ix, col in enumerate(frame.columns):
            width = {"Title":58,"Authors":40,"Author Affiliations":65,"Reason":42,
                     "Program Key":62,"Program to Assign":62,"PubMed Link":43,"DOI Link":43}.get(col,22)
            ws.set_column(ix,ix,width,wrap)
            ws.write(0,ix,col,hdr)
        ws.freeze_panes(1,0)
        ws.autofilter(0,0,max(1,len(frame)),len(frame.columns)-1)
        ws.set_zoom(80)
        if sheet_name in ("Needs Review", "Review Overflow"):
            yes_ix = list(frame.columns).index("Include?")
            prog_ix = list(frame.columns).index("Program to Assign")
            ws.data_validation(1,yes_ix,max(1000,len(frame)+10),yes_ix,
                               {"validate":"list","source":["Yes","No"]})
            ws.data_validation(1,prog_ix,max(1000,len(frame)+10),prog_ix,
                               {"validate":"list","source":"=ProgramOptions"})
print("Excel review created:", excel_path)

Excel review created: /Users/richardsavel/Library/CloudStorage/Dropbox/python test files/pubmed newsletter/kothari/kothari GME approach/2026 10 19 files/rwj_gme_pubmed_v18_2026-08-01_to_2026-09-30_2026-10-21_v02_FOR_REVIEW.xlsx


## 6 — Preliminary Word draft

Accepted automatically matched publications only. One citation per program and PMID, listing every recognized trainee, with relevant authors bolded. Includes two explicit page breaks, compressed citation spacing, clickable **PubMed**/**Article**, and a trainee-by-publication table appendix.


In [23]:
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.opc.constants import RELATIONSHIP_TYPE as RT
from docx.enum.text import WD_BREAK
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import RGBColor
from docx.enum.table import WD_ROW_HEIGHT_RULE


def add_hyperlink(paragraph, label, url):
    """Clickable Word hyperlink, not a blue string pretending to be a link."""
    if not str(url).strip():
        return
    link_id = paragraph.part.relate_to(str(url), RT.HYPERLINK, is_external=True)
    link = OxmlElement("w:hyperlink")
    link.set(qn("r:id"), link_id)
    run = OxmlElement("w:r")
    props = OxmlElement("w:rPr")
    color = OxmlElement("w:color")
    color.set(qn("w:val"), "0563C1")
    props.append(color)
    underline = OxmlElement("w:u")
    underline.set(qn("w:val"), "single")
    props.append(underline)
    run.append(props)
    t = OxmlElement("w:t")
    t.text = str(label)
    run.append(t)
    link.append(run)
    paragraph._p.append(link)


def program_label(program, trainee_type):
    suffix = "Fellowship" if str(trainee_type).strip().casefold() == "fellow" else "Residency"
    label = str(program).strip()
    if label.casefold() == "other":
        return "Other"
    return label if label.casefold().endswith(suffix.casefold()) else f"{label} {suffix}"


def tight(paragraph, after=0, before=0, keep_next=False):
    f = paragraph.paragraph_format
    f.space_before = Pt(before)
    f.space_after = Pt(after)
    f.line_spacing = 0.95
    f.keep_with_next = keep_next
    return paragraph


def department_label(name):
    name = str(name).strip()
    return name if name.casefold().startswith("department of ") else f"Department of {name}"


def set_cell_fill(cell, hex_color):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = tcPr.find(qn("w:shd"))
    if shd is None:
        shd = OxmlElement("w:shd")
        tcPr.append(shd)
    shd.set(qn("w:fill"), hex_color)


def set_cell_margins(cell, top=70, start=100, bottom=70, end=100):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    mar = tcPr.find(qn("w:tcMar"))
    if mar is None:
        mar = OxmlElement("w:tcMar")
        tcPr.append(mar)
    for side, amount in (("top",top),("start",start),("bottom",bottom),("end",end)):
        el = mar.find(qn("w:"+side))
        if el is None:
            el=OxmlElement("w:"+side); mar.append(el)
        el.set(qn("w:w"),str(amount)); el.set(qn("w:type"),"dxa")


def table_compact(table, widths, font_size=8.5):
    """Designed fixed-width table: exact columns, navy header, alternate pale-blue rows."""
    table.autofit = False
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.style = "Table Grid"
    for row_number, row in enumerate(table.rows):
        for j, cell in enumerate(row.cells):
            cell.width=Inches(widths[j])
            cell.vertical_alignment=WD_CELL_VERTICAL_ALIGNMENT.CENTER
            set_cell_margins(cell, top=55, start=95, bottom=55, end=95)
            if row_number==0:
                set_cell_fill(cell,"17365D")
            elif row_number%2==0:
                set_cell_fill(cell,"EEF3F8")
            for paragraph in cell.paragraphs:
                tight(paragraph)
                paragraph.alignment=WD_ALIGN_PARAGRAPH.CENTER if (j==len(widths)-1 and len(widths)==4) else WD_ALIGN_PARAGRAPH.LEFT
                for run in paragraph.runs:
                    run.font.name=FONT_NAME
                    run.font.size=Pt(font_size)
                    if row_number==0:
                        run.bold=True;run.font.color.rgb=RGBColor(255,255,255)
        if row_number == 0:
            trPr=row._tr.get_or_add_trPr()
            repeat=OxmlElement("w:tblHeader"); repeat.set(qn("w:val"),"true"); trPr.append(repeat)
    # Set the table grid explicitly; otherwise Word may ignore the cell widths.
    for col,width in zip(table.columns,widths):
        col.width=Inches(width)
    # Avoid ugly whole-table row splits across a page.
    for row in table.rows:
        trPr=row._tr.get_or_add_trPr()
        cant=OxmlElement("w:cantSplit")
        trPr.append(cant)


def styled_heading(doc, label, level, before=8, after=3):
    par=doc.add_heading(str(label),level=level)
    tight(par,before=before,after=after,keep_next=True)
    par.paragraph_format.left_indent=Inches(0)
    par.paragraph_format.first_line_indent=Inches(0)
    par.paragraph_format.right_indent=Inches(0)
    par.paragraph_format.keep_with_next=True
    return par


def institution_heading(doc, label, before=12, after=6):
    """Centered institutional banner; its bottom rule is independent of program rules."""
    p = styled_heading(doc, label, level=1, before=before, after=after)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    for run in p.runs:
        run.font.name = FONT_NAME
        run.font.size = Pt(15)
        run.bold = True
        run.font.color.rgb = RGBColor.from_string("17365D")
    ppr = p._p.get_or_add_pPr()
    borders = ppr.find(qn("w:pBdr"))
    if borders is None:
        borders = OxmlElement("w:pBdr")
        ppr.append(borders)
    bottom = OxmlElement("w:bottom")
    for key, value in {"val": "single", "sz": "5", "space": "5", "color": "B7C8DA"}.items():
        bottom.set(qn("w:" + key), value)
    borders.append(bottom)
    return p


def program_rule(paragraph):
    """Thin upper rule to separate training programs without adding blank paragraphs."""
    ppr = paragraph._p.get_or_add_pPr()
    borders = ppr.find(qn("w:pBdr"))
    if borders is None:
        borders = OxmlElement("w:pBdr")
        ppr.append(borders)
    rule = OxmlElement("w:top")
    for key, value in {"val":"single", "sz":"5", "space":"5", "color":"CCD9E7"}.items():
        rule.set(qn("w:" + key), value)
    borders.append(rule)


def program_heading(doc, label, before=8, after=4):
    p = styled_heading(doc, label, level=3, before=before, after=after)
    p.style = doc.styles["Heading 3"]
    program_rule(p)
    return p


def article_names(authors_string, matched_names):
    # Bold matched PubMed author names, using the matched author actually in
    # the article, rather than attempting to reconstruct their name from roster.
    matched = {" ".join(words(x)) for x in matched_names if str(x).strip()}
    return [a.strip() for a in str(authors_string).split(";") if a.strip()], matched


def add_field(paragraph, field_name):
    """Live Word field (updated by Word when opened or printed)."""
    run = paragraph.add_run()
    begin = OxmlElement("w:fldChar")
    begin.set(qn("w:fldCharType"), "begin")
    inst = OxmlElement("w:instrText")
    inst.set(qn("xml:space"), "preserve")
    inst.text = " " + field_name + " "
    separate = OxmlElement("w:fldChar")
    separate.set(qn("w:fldCharType"), "separate")
    display = OxmlElement("w:t")
    display.text = "1"
    finish = OxmlElement("w:fldChar")
    finish.set(qn("w:fldCharType"), "end")
    for element in [begin, inst, separate, display, finish]:
        run._r.append(element)
    run.font.size = Pt(8)
    run.font.color.rgb = RGBColor.from_string("61758B")

def add_page_number_footer(section):
    footer = section.footer
    footer.distance = Inches(0.38)
    p = footer.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(0)
    run = p.add_run("Page ")
    run.font.size = Pt(8)
    run.font.color.rgb = RGBColor.from_string("61758B")
    add_field(p, "PAGE")
    run = p.add_run(" of ")
    run.font.size = Pt(8)
    run.font.color.rgb = RGBColor.from_string("61758B")
    add_field(p, "NUMPAGES")

def department_heading(doc, name, before=9, after=5):
    p = styled_heading(doc, department_label(name), level=2, before=before, after=after)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.left_indent = Inches(0)
    for run in p.runs:
        run.font.name = FONT_NAME
        run.font.size = Pt(12)
        run.font.bold = True
        run.font.color.rgb = RGBColor.from_string("28628A")
    return p

def build_newsletter(accepted, output_path):
    required = ["PMID", "Program Key", "Trainee Name", "Matched Author", "Home Institution", "Department",
                "Program", "Trainee Type", "Title", "Authors", "Journal", "Publication Date", "PubMed Link", "DOI"]
    if accepted.empty:
        accepted = pd.DataFrame(columns=required)
    accepted = accepted.copy()
    for col in required:
        if col not in accepted.columns:
            accepted[col] = ""
    accepted["PMID"] = accepted["PMID"].astype(str).str.replace(r"\.0$", "", regex=True)
    accepted = accepted.drop_duplicates(subset=["PMID", "Trainee Name", "Program Key"])
    accepted["Display Program"] = [program_label(r["Program"], r["Trainee Type"])
                                   for r in accepted.to_dict("records")]
    accepted = accepted.sort_values(["Home Institution", "Department", "Display Program", "Publication Date", "Title", "Trainee Name"],
                                    ascending=[True, True, True, False, True, True], kind="stable")
    # Unique papers by program for body, but keep every resident-paper row for
    # the contributor list and appendix.
    groups = list(accepted.groupby(["Home Institution", "Department", "Display Program", "PMID"], sort=False))
    overview = (accepted.groupby(["Home Institution", "Department", "Display Program"], sort=True)["PMID"]
                .nunique().reset_index(name="Papers"))
    doc = Document()
    section = doc.sections[0]
    section.top_margin = Inches(PAGE_MARGIN_INCHES)
    section.bottom_margin = Inches(PAGE_MARGIN_INCHES)
    section.left_margin = Inches(PAGE_MARGIN_INCHES)
    section.right_margin = Inches(PAGE_MARGIN_INCHES)
    add_page_number_footer(section)
    normal = doc.styles["Normal"]
    normal.font.name = FONT_NAME
    normal.font.size = Pt(FONT_SIZE)
    normal.paragraph_format.space_after = Pt(0)
    normal.paragraph_format.line_spacing = 1.0
    for style_name in ("Heading 1", "Heading 2", "Heading 3"):
        st = doc.styles[style_name]
        st.paragraph_format.space_before = Pt(7)
        st.paragraph_format.space_after = Pt(2)
        st.paragraph_format.keep_with_next = True

    for style_name, size, color in (("Heading 1", 12, "17365D"),
                                    ("Heading 2", 10, "28628A"),
                                    ("Heading 3", 10.5, "253C52")):
        st = doc.styles[style_name]
        st.font.name = FONT_NAME
        st.font.size = Pt(size)
        st.font.bold = True
        st.font.color.rgb = RGBColor.from_string(color)
    # Disable Word widow/orphan effects from disrupting compact entries.
    normal.paragraph_format.widow_control = True

    tight(doc.add_heading(NEWSLETTER_TITLE, level=0), after=4)
    tight(doc.add_paragraph(f"Articles added to PubMed: {month_start:%B %d, %Y} – {month_end:%B %d, %Y}"), after=2)
    tight(doc.add_paragraph(f"Included publications: {accepted['PMID'].nunique()} unique papers | "
                            f"Programs with publications: {accepted['Program Key'].nunique()}"), after=5)
    styled_heading(doc, "Programs with Publications", level=1, before=9, after=6)
    summary = doc.add_table(rows=1, cols=4)
    for cell, heading in zip(summary.rows[0].cells, ["Home institution", "Department", "Program", "Papers"]):
        cell.text = heading
    for rec in overview.itertuples(index=False, name=None):
        cells = summary.add_row().cells
        for col_index, (cell, val) in enumerate(zip(cells, rec)):
            cell.text = department_label(val) if col_index == 1 else str(val)
    table_compact(summary, [2.08, 1.63, 2.70, 0.55], font_size=8.5)
    # Explicit page break after the introduction.
    doc.add_page_break()

    last_inst = last_dep = last_prog = None
    for (inst, dep, prog, pmid), rows in groups:
        rec = rows.iloc[0]
        if inst != last_inst:
            institution_heading(doc, inst, before=12, after=6)
            last_inst, last_dep, last_prog = inst, None, None
        if dep != last_dep:
            department_heading(doc, dep, before=7, after=4)
            last_dep, last_prog = dep, None
        if prog != last_prog:
            program_heading(doc, prog, before=9, after=4)
            last_prog = prog
        title = tight(doc.add_paragraph(), before=4 if last_prog == prog else 2, after=0, keep_next=True)
        title.add_run(str(rec["Title"])).bold = True
        title.paragraph_format.left_indent = Inches(0)
        authors, matched_names = article_names(rec["Authors"], rows["Matched Author"].tolist())
        par = tight(doc.add_paragraph(), after=0, keep_next=True)
        par.paragraph_format.left_indent = Inches(0)
        for ix, name in enumerate(authors):
            if ix:
                par.add_run("; ")
            run = par.add_run(name)
            run.bold = " ".join(words(name)) in matched_names
        journal_line = tight(doc.add_paragraph(), after=0, keep_next=True)
        journal_line.paragraph_format.left_indent = Inches(0)
        journal_line.add_run("Journal: ").bold = True
        journal_line.add_run(f"{rec['Journal']} ({rec['Publication Date']})")
        citation = tight(doc.add_paragraph(), after=0, keep_next=True)
        citation.paragraph_format.left_indent = Inches(0)
        citation.add_run(f"PMID: {pmid} | ")
        add_hyperlink(citation, "PubMed", rec["PubMed Link"])
        doi = str(rec.get("DOI", "") or "").strip()
        if doi and doi.lower() != "nan":
            citation.add_run(" | ")
            add_hyperlink(citation, "Article", f"https://doi.org/{doi}")
        for ix, trainee in enumerate(rows.sort_values("Trainee Name")["Trainee Name"].tolist()):
            trainee_p = tight(doc.add_paragraph(), after=3 if ix == len(rows)-1 else 0,
                              keep_next=(ix < len(rows)-1))
            trainee_p.paragraph_format.left_indent = Inches(0)
            trainee_p.add_run(f"Trainee: {trainee}")

    # Explicit page break before the complete individual-contribution appendix.
    doc.add_page_break()
    styled_heading(doc, "Trainee Publications by Program", level=1, before=3, after=5)
    tight(doc.add_paragraph("Each row represents one trainee–publication contribution. "
                            "A paper with multiple trainee authors appears once for each trainee."), after=5)
    last_key = None
    current_table = None
    for record in accepted.to_dict("records"):
        key = (record["Home Institution"], record["Department"], record["Display Program"])
        if key != last_key:
            institution_heading(doc, key[0], before=12, after=5) if last_key is None or key[0] != last_key[0] else None
            department_heading(doc, key[1], before=7, after=4) if last_key is None or key[:2] != last_key[:2] else None
            program_heading(doc, key[2], before=8, after=5)
            current_table = doc.add_table(rows=1, cols=3)
            for cell, heading in zip(current_table.rows[0].cells, ["Resident/Fellow", "Article title", "PubMed ID"]):
                cell.text = heading
            last_key = key
        cells = current_table.add_row().cells
        cells[0].text = str(record["Trainee Name"])
        cells[1].text = str(record["Title"])
        cells[2].text = ""
        add_hyperlink(cells[2].paragraphs[0], str(record["PMID"]), record["PubMed Link"])
    for table in doc.tables[1:]:
        table_compact(table, [1.67, 4.38, 0.91], font_size=8.0)
    doc.save(output_path)
    return len(overview), accepted["PMID"].nunique()

draft_path = OUTPUT_FOLDER / f"{run_stem}_DRAFT.docx"
count_programs, count_papers = build_newsletter(matched_df, draft_path)
print(f"Draft Word report: {draft_path} ({count_programs} programs, {count_papers} unique papers)")


Draft Word report: /Users/richardsavel/Library/CloudStorage/Dropbox/python test files/pubmed newsletter/kothari/kothari GME approach/2026 10 19 files/rwj_gme_pubmed_v18_2026-08-01_to_2026-09-30_2026-10-21_v02_DRAFT.docx (22 programs, 50 unique papers)


## 7 — Final reports after Excel review

Save and close your internal review workbook, then run **only the final cell**. The final cell reads automatic matches and any approved review decisions and generates **two outputs** from the same accepted rows:

1. Final Word newsletter.
2. **One-sheet, filterable Excel publication index** for Dr. Kothari and program directors. Filter by site, department, residency/fellowship program, or trainee. Each trainee–publication pair has its own row, including coauthors of the same article. This is distinct from the internal review workbook.

Both reports can be regenerated after changes to review decisions.

The file ending **`_FOR_REVIEW.xlsx`** is your internal decision workbook.
The separate **`_PUBLICATIONS.xlsx`** file is the shareable, filterable report for Dr. Kothari.

To make a PDF, open the final Word newsletter in Microsoft Word and save or export it as PDF.


In [25]:
# FINAL EXPORT — run ONLY this cell after saving and closing the reviewed Excel workbook.
from pathlib import Path
import pandas as pd

if "excel_path" in globals() and Path(excel_path).exists():
    review_excel = Path(excel_path)
else:
    possible = list(Path.cwd().glob("rwj_gme_pubmed_v18_*_v??_FOR_REVIEW.xlsx"))
    if not possible:
        raise FileNotFoundError("Run the search cells first to create an Excel review file")
    review_excel = max(possible, key=lambda p:p.stat().st_mtime)

accepted = pd.read_excel(review_excel, sheet_name="Matched Articles", dtype={"PMID":str}).fillna("")
review = pd.concat([
    pd.read_excel(review_excel, sheet_name=sheet, dtype={"PMID":str}).fillna("")
    for sheet in ("Needs Review", "Review Overflow")
], ignore_index=True)
for col in ("Include?","Program to Assign"):
    if col not in review.columns:
        raise ValueError(f"Needs Review sheet is missing {col}")
yes = review[review["Include?"].astype(str).str.strip().str.casefold().eq("yes")].copy()
invalid = yes[~yes["Program to Assign"].astype(str).str.strip().isin(REVIEW_ASSIGNMENT_CHOICES)]
if not invalid.empty:
    raise ValueError(f"{len(invalid)} Yes rows lack a valid Program to Assign. Fix and save the Excel file first.")
for index, rec in yes.iterrows():
    key = str(rec["Program to Assign"]).strip()
    if key == "Other":
        # Honor the confirmed author but never invent a residency program.
        # Preserve the trainee's actual home institution and department.
        yes.at[index, "Program"] = "Other"
        yes.at[index, "Program Key"] = f"{rec['Home Institution']} | {rec['Department']} | Other"
    elif key == "Multiple Hospitals":
        # Explicitly flag a shared-hospital attribution instead of forcing a
        # single-site assignment. Retain the trainee's program and department.
        # An independently identified trainee at another home site still has
        # their own row and can be credited there as well.
        yes.at[index, "Home Institution"] = "Multiple Hospitals"
        yes.at[index, "Program Key"] = f"Multiple Hospitals | {rec['Department']} | {rec['Program']}"
    else:
        yes.at[index,"Program Key"] = key
        for field, val in PROGRAM_LOOKUP[key].items():
            yes.at[index,field] = val
to_publish = pd.concat([accepted,yes],ignore_index=True).drop_duplicates(
    subset=["PMID", "Trainee Name", "Program Key"], keep="first"
)
output_stem = review_excel.stem.removesuffix("_FOR_REVIEW")
final_path = review_excel.with_name(output_stem + "_FINAL.docx")
program_count, publication_count = build_newsletter(to_publish, final_path)
# Additional shareable workbook — all accepted contributions, one flat filterable sheet.
# Note: Keep the private manual review workbook separate from this distribution file.
from datetime import datetime
from xlsxwriter.utility import xl_col_to_name

index_path = review_excel.with_name(output_stem + "_PUBLICATIONS.xlsx")
index = to_publish.copy()
required_index_fields = ["Home Institution", "Department", "Program", "Trainee Type", "Trainee Name",
                         "Title", "PMID", "Publication Date", "Journal", "PubMed Link", "DOI"]
for field in required_index_fields:
    if field not in index.columns:
        index[field] = ""
index["PMID"] = index["PMID"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
index["Program"] = [program_label(rec["Program"], rec["Trainee Type"]) for rec in index.to_dict("records")]
index = index.drop_duplicates(subset=["Home Institution", "Department", "Program", "Trainee Name", "PMID"])
index = index.sort_values(["Home Institution", "Department", "Program", "Trainee Name", "Publication Date", "Title"],
                          ascending=[True, True, True, True, False, True], kind="stable")
index = index.rename(columns={"Home Institution":"Site", "Trainee Name":"Trainee", "Title":"Article Title", "PMID":"PubMed ID"})
columns = ["Site", "Department", "Program", "Trainee", "Article Title", "PubMed ID", "Publication Date", "Journal", "PubMed Link"]
index = index[columns].fillna("")
# Keep identifiers as text, including leading zeros if any. Hyperlinks as real Excel hyperlinks.
with pd.ExcelWriter(index_path, engine="xlsxwriter", engine_kwargs={"options":{"strings_to_urls":False}}) as writer:
    index.to_excel(writer, sheet_name="Trainee Publications", index=False)
    book = writer.book
    worksheet = writer.sheets["Trainee Publications"]
    fmt_id = book.add_format({"num_format":"@", "valign":"top"})
    fmt_text = book.add_format({"valign":"top"})
    fmt_link = book.add_format({"font_color":"#0563C1", "underline":1})
    worksheet.freeze_panes(1, 0)
    worksheet.set_zoom(90)
    for col_number, col_name in enumerate(columns):
        width = {"Site":31, "Department":24, "Program":37, "Trainee":27,
                 "Article Title":75, "PubMed ID":16, "Publication Date":18,
                 "Journal":36, "PubMed Link":19}[col_name]
        worksheet.set_column(col_number, col_number, width, fmt_id if col_name == "PubMed ID" else fmt_text)
    if len(index):
        # Explicit Excel Table creates familiar filter dropdowns in each header.
        worksheet.add_table(0, 0, len(index), len(columns)-1, {
            "name":"GMEPublications", "style":"Table Style Medium 2",
            "columns":[{"header":name} for name in columns]
        })
        link_col = columns.index("PubMed Link")
        for row_number, url in enumerate(index["PubMed Link"].tolist(), start=1):
            if str(url).startswith(("https://", "http://")):
                worksheet.write_url(row_number, link_col, url, fmt_link, "PubMed")
        id_col = columns.index("PubMed ID")
        for row_number, pmid in enumerate(index["PubMed ID"].tolist(), start=1):
            worksheet.write_string(row_number, id_col, str(pmid), fmt_id)
    else:
        header_format = book.add_format({"bold":True, "font_color":"white", "bg_color":"#17365D"})
        for i, name in enumerate(columns):
            worksheet.write(0, i, name, header_format)
        worksheet.autofilter(0, 0, 0, len(columns)-1)

print(f"Final Word report: {final_path}")
print(f"Filterable publication index: {index_path} ({len(index)} trainee–publication rows)")
print(f"Included {publication_count} unique papers across {program_count} programs")

# Total notebook runtime (when using Run All).
elapsed = time.perf_counter() - run_start
print(f"Total runtime: {elapsed / 60:.2f} minutes")
print(f"Total runtime: {elapsed:.1f} seconds")

Final Word report: /Users/richardsavel/Library/CloudStorage/Dropbox/python test files/pubmed newsletter/kothari/kothari GME approach/2026 10 19 files/rwj_gme_pubmed_v18_2026-08-01_to_2026-09-30_2026-10-21_v02_FINAL.docx
Filterable publication index: /Users/richardsavel/Library/CloudStorage/Dropbox/python test files/pubmed newsletter/kothari/kothari GME approach/2026 10 19 files/rwj_gme_pubmed_v18_2026-08-01_to_2026-09-30_2026-10-21_v02_PUBLICATIONS.xlsx (104 trainee–publication rows)
Included 55 unique papers across 27 programs
Total runtime: 14.17 minutes
Total runtime: 850.2 seconds
